In [1]:
import os, glob, time, random, math, re
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

DATA_ROOT = "/kaggle/input/competitions/dlp-26t2-week9-assignment"
print("DATA_ROOT:", DATA_ROOT)

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

LABEL_MAP = {
    "Amphibia": 0, "Animalia": 1, "Arachnida": 2, "Aves": 3, "Fungi": 4,
    "Insecta": 5, "Mammalia": 6, "Mollusca": 7, "Plantae": 8, "Reptilia": 9
}
IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}

Device: cuda Tesla T4
DATA_ROOT: /kaggle/input/competitions/dlp-26t2-week9-assignment


In [2]:
def find_class_dirs(train_dir):
    entries = [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]
    if len(entries) == 1 and entries[0].lower() == "train":
        train_dir = os.path.join(train_dir, entries[0])
        entries = [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]
    return train_dir, entries

TRAIN_DIR, class_dirs = find_class_dirs(TRAIN_DIR)
print("Found class folders:", sorted(class_dirs))

missing = [c for c in class_dirs if c not in LABEL_MAP]
if missing:
    print("unmapped folder names ->", missing)

train_samples = []
for cname in class_dirs:
    label = LABEL_MAP[cname]
    for p in glob.glob(os.path.join(TRAIN_DIR, cname, "*")):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
            train_samples.append((p, label))

print("Total train images found:", len(train_samples))

def find_test_dir(test_dir):
    entries = [d for d in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, d))]
    if len(entries) == 1 and entries[0].lower() == "test":
        return os.path.join(test_dir, entries[0])
    return test_dir

TEST_DIR = find_test_dir(TEST_DIR)
test_paths = sorted(glob.glob(os.path.join(TEST_DIR, "*")))
test_paths = [p for p in test_paths if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
print("Total test images found:", len(test_paths))

Found class folders: ['Amphibia', 'Animalia', 'Arachnida', 'Aves', 'Fungi', 'Insecta', 'Mammalia', 'Mollusca', 'Plantae', 'Reptilia']
Total train images found: 9999
Total test images found: 2000


In [3]:
#Datasets and Augmentation
IMG_SIZE = 176
BATCH_SIZE = 96

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.85, 1.15)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.15),
    T.RandomApply([T.RandomRotation(20)], p=0.5),
    T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.05),
    T.RandomApply([T.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.15),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

eval_tf = T.Compose([
    T.Resize(int(IMG_SIZE * 1.14)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImgDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        if label is None:
            return img, os.path.splitext(os.path.basename(path))[0]
        return img, label

train_ds = ImgDataset(train_samples, train_tf)
test_ds  = ImgDataset([(p, None) for p in test_paths], eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=4, pin_memory=True, persistent_workers=True)

In [4]:
#We define the model here
class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.fc1 = nn.Conv2d(ch, max(ch // r, 8), 1)
        self.fc2 = nn.Conv2d(max(ch // r, 8), ch, 1)

    def forward(self, x):
        s = F.adaptive_avg_pool2d(x, 1)
        s = F.relu(self.fc1(s), inplace=True)
        s = torch.sigmoid(self.fc2(s))
        return x * s

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, drop_path=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.se = SEBlock(out_ch)
        self.drop_path = drop_path

        self.shortcut = None
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        identity = x if self.shortcut is None else self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        if self.training and self.drop_path > 0:
            if torch.rand(1).item() < self.drop_path:
                return F.relu(identity)
        out = out + identity
        return F.relu(out, inplace=True)

class FloraFaunaCNN(nn.Module):
    def __init__(self, num_classes=10, widths=(64, 128, 256, 512), blocks=(2, 2, 3, 2), drop_path=0.1):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, widths[0], 3, 1, 1, bias=False),
            nn.BatchNorm2d(widths[0]),
            nn.ReLU(inplace=True),
            nn.Conv2d(widths[0], widths[0], 3, 1, 1, bias=False),
            nn.BatchNorm2d(widths[0]),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        layers = []
        in_ch = widths[0]
        for stage, (w, n) in enumerate(zip(widths, blocks)):
            for i in range(n):
                stride = 2 if (i == 0 and stage != 0) else 1
                layers.append(ResBlock(in_ch, w, stride=stride, drop_path=drop_path))
                in_ch = w
        self.stages = nn.Sequential(*layers)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(widths[-1], num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stages(x)
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        return self.fc(x)

model = FloraFaunaCNN(num_classes=10).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params/1e6:.2f}M")

Model params: 12.49M


In [5]:
#Loss, optimizer, schedule, mixup
EPOCHS = 45
LR = 3e-3
WEIGHT_DECAY = 5e-2
LABEL_SMOOTHING = 0.1
MIXUP_ALPHA = 0.2
MIXUP_PROB = 0.5

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

steps_per_epoch = len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=steps_per_epoch,
    pct_start=0.15, div_factor=10, final_div_factor=100
)

scaler = torch.cuda.amp.GradScaler()

def mixup_data(x, y, alpha):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam

def mixup_criterion(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)

/tmp/ipykernel_23/1818957330.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [6]:
#Training Loop
print(f"Training for {EPOCHS} epochs, {steps_per_epoch} steps/epoch...")
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)

        use_mixup = np.random.rand() < MIXUP_PROB
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            if use_mixup:
                mixed_imgs, ya, yb, lam = mixup_data(imgs, labels, MIXUP_ALPHA)
                outputs = model(mixed_imgs)
                loss = mixup_criterion(criterion, outputs, ya, yb, lam)
            else:
                outputs = model(imgs)
                loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    elapsed = (time.time() - start_time) / 60
    cur_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | loss {epoch_loss:.4f} | train_acc {epoch_acc:.4f} "
          f"| lr {cur_lr:.5f} | elapsed {elapsed:.1f} min")

print(f"Training complete in {(time.time()-start_time)/60:.1f} minutes.")
torch.save(model.state_dict(), "/kaggle/working/flora_fauna_cnn.pt")

Training for 45 epochs, 104 steps/epoch...


/tmp/ipykernel_23/3365885784.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 01/45 | loss 2.2201 | train_acc 0.1826 | lr 0.00044 | elapsed 1.3 min
Epoch 02/45 | loss 2.1747 | train_acc 0.1971 | lr 0.00085 | elapsed 2.3 min
Epoch 03/45 | loss 2.1670 | train_acc 0.2032 | lr 0.00142 | elapsed 3.4 min
Epoch 04/45 | loss 2.1474 | train_acc 0.2160 | lr 0.00204 | elapsed 4.4 min
Epoch 05/45 | loss 2.1387 | train_acc 0.2153 | lr 0.00258 | elapsed 5.4 min
Epoch 06/45 | loss 2.1292 | train_acc 0.2258 | lr 0.00292 | elapsed 6.5 min
Epoch 07/45 | loss 2.1149 | train_acc 0.2297 | lr 0.00300 | elapsed 7.5 min
Epoch 08/45 | loss 2.0945 | train_acc 0.2343 | lr 0.00299 | elapsed 8.6 min
Epoch 09/45 | loss 2.0777 | train_acc 0.2446 | lr 0.00297 | elapsed 9.6 min
Epoch 10/45 | loss 2.0582 | train_acc 0.2514 | lr 0.00295 | elapsed 10.7 min
Epoch 11/45 | loss 2.0289 | train_acc 0.2607 | lr 0.00291 | elapsed 11.7 min
Epoch 12/45 | loss 2.0207 | train_acc 0.2760 | lr 0.00286 | elapsed 12.8 min
Epoch 13/45 | loss 1.9973 | train_acc 0.2791 | lr 0.00281 | elapsed 13.8 min
Epoch 14

In [7]:
#Inference with TTA
model.eval()
all_ids, all_probs = [], []

with torch.no_grad():
    for imgs, ids in test_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast():
            logits1 = model(imgs)
            logits2 = model(torch.flip(imgs, dims=[3]))
            probs = (F.softmax(logits1, dim=1) + F.softmax(logits2, dim=1)) / 2
        all_probs.append(probs.float().cpu())
        all_ids.extend(ids)

all_probs = torch.cat(all_probs, dim=0)
preds = all_probs.argmax(1).numpy()

/tmp/ipykernel_23/2654376681.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


In [8]:
# final prediction submission
submission = pd.DataFrame({"Image_ID": all_ids, "Label": preds})
submission = submission.sort_values("Image_ID").reset_index(drop=True)

submission.to_csv("/kaggle/working/submission.csv", index=False)
print(submission.head())

     Image_ID  Label
0  Image_0001      0
1  Image_0002      4
2  Image_0003      5
3  Image_0004      7
4  Image_0005      4
